# Shortcut Learning Risky Intent Classification

This notebook documents the full experiment flow for the shortcut-learning-risky-intent project.

We compare:

- E1: RoBERTa baseline
- E5: RoBERTa + Counterfactual Augmentation + Keyword Masking + Confidence Regularization

The goal is to test whether mitigation reduces shortcut reliance on words such as `die`, `kill`, `cut`, `jump`, and `hurt`.


## 1. Project Setup

In [ ]:
# Run this in Google Colab after cloning the repository.
# Example:
# !git clone https://github.com/<username>/shortcut-learning-risky-intent.git
# %cd shortcut-learning-risky-intent

!pip install -r requirements.txt


## 2. Data Preparation

In [ ]:
!python -m src.data_utils

## 3. Create Mitigation Datasets

In [ ]:
!python -m src.masking
!python -m src.augmentation

## 4. Train TF-IDF Baseline

In [ ]:
!python -m src.train_tfidf --experiment_id E0

## 5. Train RoBERTa Models

In [ ]:
# E1: RoBERTa baseline
!python -m src.train_roberta --experiment_id E1 --train_file data/processed/train.csv

# E5: RoBERTa + Counterfactual + Keyword Masking + Confidence Regularization
!python -m src.train_roberta --experiment_id E5 --train_file data/processed/train_full.csv --confidence_regularization true --lambda_conf 0.05


## 6. Evaluate ID and OOD

In [ ]:
!python -m src.evaluate --experiment_id E1
!python -m src.evaluate --experiment_id E5


## 7. Shortcut Sensitivity Analysis

In [ ]:
!python -m src.shortcut_metrics

## 8. MC Dropout Uncertainty Analysis

In [ ]:
!python -m src.mc_dropout --experiment_id E1 --mc_passes 30
!python -m src.mc_dropout --experiment_id E5 --mc_passes 30


## 9. SHAP Analysis

In [ ]:
!python -m src.shap_analysis --experiment_id E1
!python -m src.shap_analysis --experiment_id E5


## 10. Generate Plots

In [ ]:
!python -m src.plot_utils

## 11. Report Notes

In [ ]:
import pandas as pd
from pathlib import Path

# Load shortcut sensitivity summary
shortcut_path = Path("results/shortcut_metrics/shortcut_summary.csv")
if shortcut_path.exists():
    display(pd.read_csv(shortcut_path))

# Load SHAP summary
shap_path = Path("results/shap/shap_summary.csv")
if shap_path.exists():
    display(pd.read_csv(shap_path))

# Show available plots
plot_dir = Path("results/plots")
if plot_dir.exists():
    for plot in plot_dir.glob("*.png"):
        print(plot)


## Interpretation Guide

Use these points in the final report:

1. If E5 has better OOD macro-F1 than E1, the mitigation improves robustness.
2. If E5 has a smaller ID-OOD gap, the final model generalizes better.
3. If E5 has lower Keyword Sensitivity Score, the model is less dependent on shortcut keywords.
4. If E5 has lower Prediction Flip Rate after masking, predictions are more stable.
5. If E5 has lower SHAP keyword reliance, shortcut tokens contribute less to the prediction.
6. If MC Dropout uncertainty is higher for difficult OOD examples, uncertainty analysis is useful for risk detection.
